## Install

We install `gradio` and `transformers`

In [1]:
!pip install gradio
!pip install transformers

## VQA App

This code design VQA bot.

I'll create a Gradio app using the BLIP VQA model from Hugging Face. Here's a complete implementation:Here's also a more minimal version if you prefer something simpler:## Setup Instructions

**Install required packages:**
```bash
pip install gradio torch transformers pillow accelerate
```

**Key Features:**
- Uses Salesforce's BLIP VQA model (base version for faster inference)
- Automatic GPU detection and usage
- Error handling for edge cases
- Clean, professional UI with examples

**Alternative Models you can swap in:**
- `"Salesforce/blip-vqa-capfilt-large"` - More accurate but slower
- `"dandelin/vilt-b32-finetuned-vqa"` - ViLT-based alternative
- `"microsoft/git-base-vqav2"` - GIT model for VQA

**Performance Notes:**
- First run will download ~990MB model files
- GPU recommended but works on CPU
- Typical inference time: 1-3 seconds per question

The full version includes better UX with examples, error handling, and styling. The minimal version is just 20 lines and perfect for quick prototyping or embedding in larger systems.

Both apps will create a shareable public link when launched. Given your background with production ML systems, you'll probably want to add authentication, rate limiting, and proper logging for any production deployment.

In [2]:
import gradio as gr
import torch
from transformers import BlipProcessor, BlipForQuestionAnswering

# Load model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

def vqa_answer(image, question):
    if image is None or not question.strip():
        return "Please provide both an image and a question."

    inputs = processor(image, question, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=50)
    answer = processor.decode(outputs[0], skip_special_tokens=True)
    return answer

# Create interface
demo = gr.Interface(
    fn=vqa_answer,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Textbox(label="Question", placeholder="Ask about the image...")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="Visual Question Answering",
    description="Upload an image and ask a question about it!"
)

if __name__ == "__main__":
    demo.launch(share=True)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6179b1130242dd09b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import gradio as gr
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

# Load your fine-tuned model and processor
processor = AutoProcessor.from_pretrained("eagle0504/blip-vqa-finetuned-draft-1")
model = AutoModelForVision2Seq.from_pretrained("eagle0504/blip-vqa-finetuned-draft-1")

def generate_qa_pair(image):
    if image is None:
        return "Please provide an image.", "", ""

    try:
        # Process the image
        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs.pixel_values

        # Generate Q&A pair
        with torch.no_grad():
            generated_ids = model.generate(pixel_values=pixel_values, max_length=50)
            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Parse the structured output with flexible spacing
        if "question" in generated_text and "answer" in generated_text:
            # Handle the spaces in tags: < question > and < / question >
            import re

            # Extract question with regex to handle spaces
            question_pattern = r'< question >(.*?)< / question >'
            question_match = re.search(question_pattern, generated_text)

            # Extract answer with regex to handle spaces
            answer_pattern = r'< answer >(.*?)< / answer >'
            answer_match = re.search(answer_pattern, generated_text)

            if question_match and answer_match:
                question = question_match.group(1).strip()
                answer = answer_match.group(1).strip()
                return generated_text, question, answer
            else:
                return generated_text, "Could not parse question", "Could not parse answer"
        else:
            return generated_text, "No question/answer tags found", "No question/answer tags found"

    except Exception as e:
        return f"Error: {str(e)}", "", ""

# Create interface
demo = gr.Interface(
    fn=generate_qa_pair,
    inputs=[
        gr.Image(type="pil", label="Upload Image")
    ],
    outputs=[
        gr.Textbox(label="Full Generated Output", lines=3),
        gr.Textbox(label="Extracted Question"),
        gr.Textbox(label="Extracted Answer")
    ],
    title="BLIP VQA Fine-tuned - Question-Answer Generation",
    description="Upload an image and the model will generate a relevant question-answer pair about it!",
    examples=[
        # You can add example images here if you have some
        # ["path/to/example1.jpg"],
        # ["path/to/example2.jpg"],
    ]
)

if __name__ == "__main__":
    demo.launch(share=True)

preprocessor_config.json:   0%|          | 0.00/431 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/modeling_auto.py:2160: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24174b5874371b311a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
